In [2]:
import pandas as pd
import numpy as np
import os
import requests
from bs4 import BeautifulSoup
import time

data_path = os.path.expanduser('~/Desktop/football-intelligence/data/')

# Chargement de TOUTES les apparences (pas seulement Top 5)
appearances = pd.read_csv(data_path + 'appearances.csv')
players = pd.read_csv(data_path + 'players.csv')
valuations = pd.read_csv(data_path + 'player_valuations.csv')

print(f"✅ Apparences totales : {len(appearances):,}")
print(f"✅ Joueurs : {len(players):,}")
print(f"\nColonnes apparences : {list(appearances.columns)}")

✅ Apparences totales : 1,894,350
✅ Joueurs : 50,149

Colonnes apparences : ['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']


In [3]:
# ============================================
# CALCUL DES STATS COMPLÈTES PAR JOUEUR
# Toutes compétitions confondues
# ============================================

# Stats globales par joueur (toutes compétitions)
stats_global = appearances.groupby('player_id').agg(
    total_appearances=('appearance_id', 'count'),
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    total_yellow=('yellow_cards', 'sum'),
    total_red=('red_cards', 'sum')
).reset_index()

# Stats sur les 12 derniers mois (forme récente)
appearances['date'] = pd.to_datetime(appearances['date'])
date_12m = pd.Timestamp.now() - pd.DateOffset(months=12)
recent = appearances[appearances['date'] >= date_12m]

stats_recent = recent.groupby('player_id').agg(
    recent_appearances=('appearance_id', 'count'),
    recent_goals=('goals', 'sum'),
    recent_assists=('assists', 'sum'),
    recent_minutes=('minutes_played', 'sum'),
).reset_index()

# Stats sur les 6 derniers mois (forme courante)
date_6m = pd.Timestamp.now() - pd.DateOffset(months=6)
recent_6m = appearances[appearances['date'] >= date_6m]

stats_6m = recent_6m.groupby('player_id').agg(
    form_appearances=('appearance_id', 'count'),
    form_goals=('goals', 'sum'),
    form_assists=('assists', 'sum'),
    form_minutes=('minutes_played', 'sum'),
).reset_index()

# Fusion de tout
player_stats_full = stats_global.merge(stats_recent, on='player_id', how='left')
player_stats_full = player_stats_full.merge(stats_6m, on='player_id', how='left')

# Calcul des ratios par 90 minutes
player_stats_full['goals_per90'] = (
    player_stats_full['total_goals'] / player_stats_full['total_minutes'] * 90
).round(2)

player_stats_full['assists_per90'] = (
    player_stats_full['total_assists'] / player_stats_full['total_minutes'] * 90
).round(2)

player_stats_full['minutes_per_game'] = (
    player_stats_full['total_minutes'] / player_stats_full['total_appearances']
).round(1)

# Stats récentes par 90 min
player_stats_full['recent_goals_per90'] = (
    player_stats_full['recent_goals'] / player_stats_full['recent_minutes'].clip(1) * 90
).round(2)

player_stats_full['recent_assists_per90'] = (
    player_stats_full['recent_assists'] / player_stats_full['recent_minutes'].clip(1) * 90
).round(2)

# Score de forme (6 derniers mois)
player_stats_full['form_score'] = (
    player_stats_full['form_goals'].fillna(0) * 0.6 +
    player_stats_full['form_assists'].fillna(0) * 0.4
).round(3)

print(f"✅ Stats complètes calculées pour {len(player_stats_full):,} joueurs")
print(f"\nExemple stats :")
print(player_stats_full[['player_id', 'total_appearances', 'total_goals', 
                           'goals_per90', 'assists_per90', 'minutes_per_game',
                           'recent_goals_per90', 'form_score']].head(5).to_string())

✅ Stats complètes calculées pour 29,531 joueurs

Exemple stats :
   player_id  total_appearances  total_goals  goals_per90  assists_per90  minutes_per_game  recent_goals_per90  form_score
0         10                136           48         0.49           0.26              64.8                 NaN         0.0
1         26                152            0         0.00           0.00              88.9                 NaN         0.0
2         65                122           38         0.39           0.13              72.0                 NaN         0.0
3         77                  4            0         0.00           0.00              76.8                 NaN         0.0
4         80                 12            0         0.00           0.00              90.0                 NaN         0.0


In [5]:
# Renommer pour éviter les conflits
latest_val = valuations.sort_values('date').groupby('player_id').last().reset_index()
latest_val = latest_val[['player_id', 'market_value_in_eur']].rename(
    columns={'market_value_in_eur': 'current_market_value'}
)

max_val = valuations.groupby('player_id')['market_value_in_eur'].max().reset_index()
max_val.columns = ['player_id', 'highest_market_value']

# Vérifier les colonnes disponibles dans players
print("Colonnes players :")
print([c for c in players.columns if 'value' in c.lower() or 'market' in c.lower()])

Colonnes players :
['market_value_in_eur', 'highest_market_value_in_eur']


In [6]:
# ============================================
# FUSION CORRECTE — sans conflit de colonnes
# ============================================

# Fusion stats complètes + infos joueurs
players_corrected = players.merge(
    player_stats_full, on='player_id', how='left'
)

# Calcul de l'âge
players_corrected['date_of_birth'] = pd.to_datetime(
    players_corrected['date_of_birth'], errors='coerce'
)
players_corrected['age'] = players_corrected['date_of_birth'].apply(
    lambda x: int((pd.Timestamp.now() - x).days / 365.25) if pd.notna(x) else np.nan
)

# Scores ML
players_corrected['offensive_score'] = (
    players_corrected['goals_per90'].fillna(0) * 0.6 +
    players_corrected['assists_per90'].fillna(0) * 0.4
).round(3)

players_corrected['regularity_score'] = (
    players_corrected['minutes_per_game'].fillna(0) / 90
).clip(0, 1).round(3)

players_corrected['discipline_score'] = (
    1 - (players_corrected['total_yellow'].fillna(0) * 0.02 +
         players_corrected['total_red'].fillna(0) * 0.1) /
    players_corrected['total_appearances'].fillna(1).clip(1)
).clip(0, 1).round(3)

players_corrected['potential_ratio'] = (
    players_corrected['market_value_in_eur'].fillna(0) /
    players_corrected['highest_market_value_in_eur'].clip(1)
).clip(0, 1).round(3)

# Filtre joueurs valides
players_valid = players_corrected[
    players_corrected['market_value_in_eur'].notna() &
    players_corrected['total_appearances'].notna() &
    players_corrected['age'].notna()
].copy()

print(f"✅ Joueurs valides : {len(players_valid):,}")

# Vérification Mbappé
mbappe = players_valid[players_valid['name'].str.contains('Mbapp', na=False)]
if len(mbappe) > 0:
    m = mbappe.iloc[0]
    print(f"\n✅ Vérification Mbappé :")
    print(f"  Matchs totaux : {int(m['total_appearances'])}")
    print(f"  Buts totaux : {int(m['total_goals'])}")
    print(f"  Buts/90 : {m['goals_per90']}")
    print(f"  Passes/90 : {m['assists_per90']}")
    print(f"  Minutes/match : {m['minutes_per_game']}")
    print(f"  Valeur : {m['market_value_in_eur']/1e6:.1f}M€")
    print(f"  Forme récente (6m) : {m['form_score']}")

✅ Joueurs valides : 25,876

✅ Vérification Mbappé :
  Matchs totaux : 429
  Buts totaux : 330
  Buts/90 : 0.9
  Passes/90 : 0.31
  Minutes/match : 76.9
  Valeur : 180.0M€
  Forme récente (6m) : 8.8


In [7]:
# ============================================
# MISE À JOUR DU MODÈLE AVANCÉ
# ============================================
import pickle

# Top 5 ligues
TOP5 = ['GB1', 'ES1', 'FR1', 'IT1', 'L1']

# Charger le modèle existant
with open(data_path + 'advanced_model.pkl', 'rb') as f:
    advanced_model = pickle.load(f)

# Filtrer joueurs Top 5 avec les nouvelles stats
clubs_top5 = advanced_model['clubs_top5']
club_ids_top5 = clubs_top5['club_id'].tolist()

players_top5_corrected = players_valid[
    players_valid['current_club_id'].isin(club_ids_top5)
].copy()

print(f"✅ Joueurs Top 5 avec stats corrigées : {len(players_top5_corrected):,}")

# Vérification colonnes disponibles
needed = ['name', 'position', 'age', 'market_value_in_eur', 
          'total_appearances', 'total_goals', 'total_assists',
          'goals_per90', 'assists_per90', 'minutes_per_game',
          'total_yellow', 'total_red', 'foot', 'offensive_score',
          'regularity_score', 'discipline_score', 'potential_ratio',
          'form_score', 'current_club_id']

available = [c for c in needed if c in players_top5_corrected.columns]
missing = [c for c in needed if c not in players_top5_corrected.columns]

print(f"✅ Colonnes disponibles : {len(available)}")
if missing:
    print(f"⚠️ Colonnes manquantes : {missing}")

✅ Joueurs Top 5 avec stats corrigées : 7,759
✅ Colonnes disponibles : 19


In [8]:
# ============================================
# SAUVEGARDE DU MODÈLE CORRIGÉ
# ============================================

# Ajouter les colonnes manquantes pour la compatibilité avec l'app
players_top5_corrected['name_player'] = players_top5_corrected['name']

# Fusion avec le nom du club
clubs_top5 = advanced_model['clubs_top5']
players_top5_corrected = players_top5_corrected.merge(
    clubs_top5[['club_id', 'name', 'domestic_competition_id']],
    left_on='current_club_id', right_on='club_id', how='left',
    suffixes=('', '_club')
)
players_top5_corrected['name_club'] = players_top5_corrected['name_club']

# Mettre à jour le modèle
advanced_model['players_top5'] = players_top5_corrected

# Sauvegarder
with open(data_path + 'advanced_model.pkl', 'wb') as f:
    pickle.dump(advanced_model, f)

print("✅ Modèle mis à jour avec stats corrigées !")
print(f"   - {len(players_top5_corrected):,} joueurs Top 5")
print(f"\nVérification Mbappé dans le nouveau modèle :")
m = players_top5_corrected[players_top5_corrected['name'].str.contains('Mbapp', na=False)]
if len(m) > 0:
    m = m.iloc[0]
    print(f"  Matchs : {int(m['total_appearances'])}")
    print(f"  Buts : {int(m['total_goals'])}")
    print(f"  Buts/90 : {m['goals_per90']}")
    print(f"  Valeur : {m['market_value_in_eur']/1e6:.1f}M€")
    print(f"  Club : {m['name_club']}")

✅ Modèle mis à jour avec stats corrigées !
   - 7,759 joueurs Top 5

Vérification Mbappé dans le nouveau modèle :
  Matchs : 429
  Buts : 330
  Buts/90 : 0.9
  Valeur : 180.0M€
  Club : Real Madrid
